[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ZeruiW/frontier-ai-courses/blob/main/C68_Eval_Infrastructure_Course/00_setup/00_environment_check.ipynb)

# 00 · 课程总览与环境（最小 eval 基础设施 / 幂等性 / 成本模型 / 四层架构自检）

目标：用 120 行代码把四层架构完整跑一遍——spec、runner、store、report 各一层，
后面五个模块都是在这个骨架上把某一层做深。

本 notebook 你会亲手实现：
1. **环境自检与临时目录约定**
2. **四层骨架** —— spec（纯数据 + 哈希）· runner（幂等）· store（只追加）· report
3. **幂等性验证** —— 重跑不产生重复行、中断后能续跑、增量扩样本安全
4. **可归因性** —— 改一个配置项，指纹就变；指纹相同而结果不同 = 告警
5. **「一次性脚本」vs「基础设施」的成本模型** —— 跑几次之后开始回本
6. **六个反模式的自检器** —— 把你现有的流程打个分

> 心智模型：**基础设施要保证的不是「跑得更快」，而是三条性质——
> 确定性（差异可解释）、可归因性（差异指向配置项）、幂等性（重跑不破坏）。**

## 0 · 环境自检与临时目录

In [ ]:
import sys, os, json, math, time, hashlib, shutil, sqlite3, random
from collections import Counter, defaultdict

import numpy as np

print('Python :', sys.version.split()[0])
print('numpy  :', np.__version__)

TMP = os.path.abspath('./_eval_tmp')
if os.path.exists(TMP):
    shutil.rmtree(TMP)                 # 先清空再开始 —— 让 notebook 自己是幂等的
os.makedirs(TMP, exist_ok=True)
print('临时目录:', TMP)
assert os.path.isdir(TMP) and not os.listdir(TMP)
print('\n✅ 环境就绪：本课全部内容 CPU 可跑、断网可跑，不需要任何 API key。')

## 1 · ① SPEC 层：纯数据，可哈希

关键约束：**spec 里不含任何执行逻辑**。它只描述「评什么、怎么跑、怎么判」，
因此可以被序列化、被哈希、被 diff。**指纹就是它的哈希** —— 不需要人去维护一份清单。

In [ ]:
def stable_hash(obj, n=8):
    """对任意可 JSON 化的对象取稳定哈希。sort_keys 保证字段顺序不影响结果。"""
    payload = json.dumps(obj, sort_keys=True, ensure_ascii=False, separators=(',', ':'))
    return hashlib.sha256(payload.encode('utf-8')).hexdigest()[:n]

SPEC = {
    'name': 'demo-qa',
    'dataset': {'id': 'qa-tasks', 'version': 'v1', 'sha': None},   # sha 稍后填
    'model': {'id': 'fake-model-a', 'temperature': 0.0, 'seed': 0},
    'scorer': {'kind': 'exact_match', 'version': 'v1'},
    'budget': {'max_attempts': 1, 'timeout_s': 5, 'retries': 2},
    'runner': {'concurrency': 4},
}

TASKS = [
    {'task_id': 'q001', 'input': '2+2', 'target': '4'},
    {'task_id': 'q002', 'input': '3*7', 'target': '21'},
    {'task_id': 'q003', 'input': '10-4', 'target': '6'},
    {'task_id': 'q004', 'input': '9/3',  'target': '3'},
    {'task_id': 'q005', 'input': '5+5',  'target': '10'},
]
SPEC['dataset']['sha'] = stable_hash(TASKS, 12)

def fingerprint(spec):
    return stable_hash(spec, 8)

fp = fingerprint(SPEC)
print('spec 指纹:', fp)
print('数据集 sha:', SPEC['dataset']['sha'])

# 字段顺序不影响哈希；改任何一个值都会改变哈希
reordered = {k: SPEC[k] for k in reversed(list(SPEC))}
assert fingerprint(reordered) == fp, '字段顺序不应影响指纹'
changed = json.loads(json.dumps(SPEC)); changed['model']['temperature'] = 0.7
assert fingerprint(changed) != fp, '改了运行指纹必须变'
print('\n✅ spec 是纯数据 → 指纹是它的哈希，不需要人去维护一份「要记录哪些字段」的清单。')
print('   这就是 C66-05 的运行指纹，只是现在它是架构上必然做到的，而不是一条纪律。')

## 2 · ② RUNNER 层：被评「模型」与幂等执行

被评对象是一个可控的假模型：确定性输出 + 可注入的失败（超时/限流/格式错误）。
这比真实 API 更适合验证 runner —— **你可以精确控制第几次调用会失败**。

In [ ]:
class FakeModel:
    """可控的被评模型。fail_plan: {调用序号: 失败类型}，用于精确注入失败。"""

    def __init__(self, model_id='fake-model-a', accuracy=1.0, fail_plan=None, seed=0):
        self.model_id = model_id
        self.accuracy = accuracy
        self.fail_plan = dict(fail_plan or {})
        self.rng = random.Random(seed)
        self.n_calls = 0

    def __call__(self, text):
        self.n_calls += 1
        if self.n_calls in self.fail_plan:
            raise RuntimeError(self.fail_plan[self.n_calls])
        try:
            correct = str(int(eval(text, {'__builtins__': {}}, {})))
        except Exception:
            correct = ''
        if self.rng.random() < self.accuracy:
            return correct
        return correct + '0'                     # 一个可控的错误答案


def exact_match(output, target):
    return 1.0 if str(output).strip() == str(target).strip() else 0.0


def run_one(spec, task, model, scorer):
    """跑一条任务，返回一行结果。所有失败都被分类，绝不静默跳过。"""
    t0 = time.time()
    retries = spec['budget']['retries']
    for attempt_i in range(retries + 1):
        try:
            out = model(task['input'])
            return {'task_id': task['task_id'], 'status': 'ok',
                    'output': out, 'score': scorer(out, task['target']),
                    'error': None, 'n_retries': attempt_i,
                    'latency_ms': int((time.time() - t0) * 1000)}
        except Exception as e:
            last = f'{type(e).__name__}: {e}'
    return {'task_id': task['task_id'], 'status': 'error',
            'output': None, 'score': None, 'error': last,
            'n_retries': retries, 'latency_ms': int((time.time() - t0) * 1000)}

model = FakeModel(accuracy=0.8, seed=1)
row = run_one(SPEC, TASKS[0], model, exact_match)
print('一行结果:', {k: row[k] for k in ('task_id', 'status', 'output', 'score', 'n_retries')})
assert set(row) >= {'task_id', 'status', 'score', 'error', 'n_retries', 'latency_ms'}
print('\n✅ 注意 status 字段：失败的样本也会产出一行（score=None），而不是被跳过。')
print('   这是反模式 5 的解法——失败必须被分类并计入，否则分母变小、成功率虚高。')

## 3 · ③ STORE 层：只追加、主键唯一、幂等写入

主键 = `(run_id, task_id, attempt)`。写入前先查在不在，在就跳过。
**这一条同时解决了三件事**：中断续跑、部分重跑、以及「不小心跑了两次」。

In [ ]:
class Store:
    """最小结果存储：SQLite，只追加，主键唯一。"""

    def __init__(self, path):
        self.conn = sqlite3.connect(path)
        self.conn.execute('''CREATE TABLE IF NOT EXISTS results (
            run_id TEXT, task_id TEXT, attempt INTEGER,
            fingerprint TEXT, status TEXT, score REAL,
            output TEXT, error TEXT, n_retries INTEGER, latency_ms INTEGER,
            PRIMARY KEY (run_id, task_id, attempt))''')
        self.conn.commit()

    def has(self, run_id, task_id, attempt):
        cur = self.conn.execute(
            'SELECT 1 FROM results WHERE run_id=? AND task_id=? AND attempt=?',
            (run_id, task_id, attempt))
        return cur.fetchone() is not None

    def put(self, run_id, attempt, fingerprint, row):
        """幂等写入：已存在就跳过，返回是否真的写了。"""
        if self.has(run_id, row['task_id'], attempt):
            return False
        self.conn.execute(
            'INSERT INTO results VALUES (?,?,?,?,?,?,?,?,?,?)',
            (run_id, row['task_id'], attempt, fingerprint, row['status'], row['score'],
             json.dumps(row['output']), row['error'], row['n_retries'], row['latency_ms']))
        self.conn.commit()
        return True

    def rows(self, run_id):
        cur = self.conn.execute(
            'SELECT task_id, attempt, fingerprint, status, score, n_retries, latency_ms '
            'FROM results WHERE run_id=? ORDER BY task_id, attempt', (run_id,))
        cols = ['task_id', 'attempt', 'fingerprint', 'status', 'score', 'n_retries', 'latency_ms']
        return [dict(zip(cols, r)) for r in cur.fetchall()]

    def count(self, run_id):
        return self.conn.execute(
            'SELECT COUNT(*) FROM results WHERE run_id=?', (run_id,)).fetchone()[0]


DB = os.path.join(TMP, 'results.db')
store = Store(DB)

def run_eval(spec, tasks, store, run_id, model_factory, scorer, stop_after=None):
    """runner 主循环：幂等 + 可中断。stop_after 用来模拟中断。"""
    fp = fingerprint(spec)
    model = model_factory()
    n_new = n_skip = 0
    for i, task in enumerate(tasks):
        for attempt in range(spec['budget']['max_attempts']):
            if store.has(run_id, task['task_id'], attempt):
                n_skip += 1
                continue
            row = run_one(spec, task, model, scorer)
            store.put(run_id, attempt, fp, row)
            n_new += 1
        if stop_after is not None and i + 1 >= stop_after:
            break                                  # 模拟中断
    return {'new': n_new, 'skipped': n_skip}

RUN_ID = 'run-2026-08-29-a'
r1 = run_eval(SPEC, TASKS, store, RUN_ID, lambda: FakeModel(accuracy=0.8, seed=1), exact_match)
print('第一次跑:', r1, '| 库里行数:', store.count(RUN_ID))
assert r1['new'] == len(TASKS) and store.count(RUN_ID) == len(TASKS)
print('\n✅ 五条任务全部跑完并落库。')

In [ ]:
# 幂等性验证 ①：重跑不产生重复行
r2 = run_eval(SPEC, TASKS, store, RUN_ID, lambda: FakeModel(accuracy=0.8, seed=1), exact_match)
print('第二次跑:', r2, '| 库里行数:', store.count(RUN_ID))
assert r2['new'] == 0 and r2['skipped'] == len(TASKS)
assert store.count(RUN_ID) == len(TASKS), '重跑不应产生任何新行'

# 幂等性验证 ②：中断后续跑
RUN_B = 'run-2026-08-29-b'
part = run_eval(SPEC, TASKS, store, RUN_B, lambda: FakeModel(accuracy=0.8, seed=2),
                exact_match, stop_after=2)
print(f'\n中断在第 2 条: 已完成 {store.count(RUN_B)} / {len(TASKS)}')
rest = run_eval(SPEC, TASKS, store, RUN_B, lambda: FakeModel(accuracy=0.8, seed=2), exact_match)
print(f'续跑后: 新增 {rest["new"]}，跳过 {rest["skipped"]}，共 {store.count(RUN_B)}')
assert store.count(RUN_B) == len(TASKS)
assert rest['new'] == len(TASKS) - 2 and rest['skipped'] == 2

# 幂等性验证 ③：增量扩样本是安全的
NEW_TASKS = TASKS + [{'task_id': 'q006', 'input': '12/4', 'target': '3'},
                     {'task_id': 'q007', 'input': '8+7',  'target': '15'}]
inc = run_eval(SPEC, NEW_TASKS, store, RUN_ID, lambda: FakeModel(accuracy=0.8, seed=1), exact_match)
print(f'\n任务集从 {len(TASKS)} 扩到 {len(NEW_TASKS)}: 新增 {inc["new"]}，跳过 {inc["skipped"]}')
assert inc['new'] == 2 and inc['skipped'] == len(TASKS)
print('\n✅ 三条幂等性全部通过：重跑不破坏 · 中断能续 · 增量扩样本只跑新增的。')
print('   实现只用了一句话：**主键 (run_id, task_id, attempt)，写入前先查在不在。**')

## 4 · ④ REPORT 层 + 可归因性

报告层本身的统计规范由 C66/C67 负责，本课不重复。
这一节只演示**可归因性**：指纹相同而结果不同 = 有未被记录的变量在动。

In [ ]:
def summarize(store, run_id):
    rows = store.rows(run_id)
    ok = [r for r in rows if r['status'] == 'ok']
    err = [r for r in rows if r['status'] != 'ok']
    scores = [r['score'] for r in ok]
    fps = {r['fingerprint'] for r in rows}
    return {
        'run_id': run_id,
        'n_total': len(rows),
        'n_ok': len(ok), 'n_error': len(err),
        # 关键：分母是**全部**样本，不是只算成功的（反模式 5）
        'score_mean': (sum(scores) / len(rows)) if rows else float('nan'),
        'score_mean_ok_only': (sum(scores) / len(ok)) if ok else float('nan'),
        'error_rate': len(err) / len(rows) if rows else 0.0,
        'p50_latency_ms': int(np.median([r['latency_ms'] for r in rows])) if rows else 0,
        'fingerprints': sorted(fps),
    }

rep = summarize(store, RUN_ID)
for k, v in rep.items():
    print(f'  {k:<22} {v}')
assert len(rep['fingerprints']) == 1, '同一个 run 的所有行必须共享同一个指纹'
print('\n注意两个分母不同的成功率：全体 vs 只算成功的。')
print('✅ 报告里必须用**全体**做分母，否则失败样本被悄悄排除，成功率虚高。')

In [ ]:
# 可归因性演示：改一个配置项 → 指纹变 → 两次运行不可直接比较
SPEC_HOT = json.loads(json.dumps(SPEC))
SPEC_HOT['model']['temperature'] = 0.7
RUN_C = 'run-2026-08-29-c'
run_eval(SPEC_HOT, TASKS, store, RUN_C, lambda: FakeModel(accuracy=0.6, seed=3), exact_match)

rep_a, rep_c = summarize(store, RUN_ID), summarize(store, RUN_C)
print(f"{'run':<22}{'指纹':>12}{'成功率':>10}")
for r in (rep_a, rep_c):
    print(f"{r['run_id']:<22}{r['fingerprints'][0]:>12}{r['score_mean']:>10.1%}")

def comparable(rep1, rep2):
    return rep1['fingerprints'] == rep2['fingerprints']

print(f'\n可直接比较吗: {comparable(rep_a, rep_c)}')
assert not comparable(rep_a, rep_c)
print('✅ 指纹不同 → 这两个数字不能直接比较，差异里混着 temperature 的影响。')
print('   **在 CI 门禁里这就是一行 assert**（04 模块），而不是靠人去回忆改过什么。')

# 反过来：指纹相同却结果差很多 = 有未被记录的变量在动
RUN_D = 'run-2026-08-29-d'
run_eval(SPEC, TASKS, store, RUN_D, lambda: FakeModel(accuracy=0.2, seed=9), exact_match)
rep_d = summarize(store, RUN_D)
gap = abs(rep_a['score_mean'] - rep_d['score_mean'])
print(f'\n同指纹的两次运行: {rep_a["score_mean"]:.0%} vs {rep_d["score_mean"]:.0%}，差 {gap:.0%}')
assert comparable(rep_a, rep_d) and gap > 0.2
print('⚠️ 指纹相同却差这么多 → **告警**：存在未被 spec 记录的变量（这里是模型的真实准确率）。')
print('   这类告警是基础设施最有价值的产出之一——它抓的是「你以为固定了但其实没有」的东西。')

## 5 · 成本模型：跑几次之后基础设施开始回本

In [ ]:
def cost_model(n_runs, script_setup=0.5, script_per_run=1.2,
               infra_setup=8.0, infra_per_run=0.15):
    """单位：人小时。script_per_run 包含手工配置、手工记录、事后拼凑上下文的时间。"""
    return (script_setup + script_per_run * n_runs,
            infra_setup + infra_per_run * n_runs)

print(f"{'跑的次数':>10}{'一次性脚本':>14}{'基础设施':>12}{'谁更划算':>12}")
for n in [1, 3, 5, 10, 20, 50, 200]:
    s, i = cost_model(n)
    print(f'{n:>10}{s:>14.1f}{i:>12.1f}{("脚本" if s < i else "基础设施"):>12}')

breakeven = next(n for n in range(1, 500) if cost_model(n)[1] < cost_model(n)[0])
print(f'\n回本点: 第 {breakeven} 次运行')
assert 3 < breakeven < 20
print('✅ 大约跑七八次之后基础设施开始回本——这与「第三次手工重复同一件事时开始建」这条经验一致。')
print('   （留了几次余量，因为早期评测的定义还在变，过早固化 schema 会变成迁移成本。）')

# 但对 CI 场景，这个账完全不同
s_ci, i_ci = cost_model(500)
print(f'\nCI 场景（每次提交都跑，一年 500 次）: 脚本 {s_ci:.0f} 人小时 vs 基础设施 {i_ci:.0f} 人小时')
assert s_ci > 5 * i_ci
print('   而且这还没算「误报让团队关掉门禁」的隐性成本——那个成本无法用人小时衡量。')

## 6 · 六个反模式的自检器

In [ ]:
ANTI_PATTERNS = [
    ('spec_is_data',    '配置是纯数据（可序列化、可哈希），不是写死在代码里'),
    ('dataset_append_only', '任务集只增不改；发现坏题时发新版本而非就地修改'),
    ('stable_task_ids', '样本 ID 稳定：改内容不改 ID，删样本不复用 ID'),
    ('cache_key_has_fp','缓存键包含完整运行指纹（模型/prompt/参数）'),
    ('failures_counted','失败样本被分类并计入分母，不是静默跳过'),
    ('ci_threshold_from_variance', 'CI 门禁阈值由重复运行的方差推出，不是拍脑袋'),
]

def audit(flow):
    passed = [k for k, _ in ANTI_PATTERNS if flow.get(k)]
    missing = [(k, d) for k, d in ANTI_PATTERNS if not flow.get(k)]
    return len(passed) / len(ANTI_PATTERNS), missing

typical_early_stage = {'spec_is_data': False, 'dataset_append_only': False,
                       'stable_task_ids': True, 'cache_key_has_fp': False,
                       'failures_counted': False, 'ci_threshold_from_variance': False}
score, missing = audit(typical_early_stage)
print(f'一个「刚从脚本长起来」的流程自检得分: {score:.0%}\n')
for k, d in missing:
    print(f'  ✗ [{k}] {d}')
assert abs(score - 1 / 6) < 1e-9

our_flow = {'spec_is_data': True, 'dataset_append_only': True, 'stable_task_ids': True,
            'cache_key_has_fp': False, 'failures_counted': True,
            'ci_threshold_from_variance': False}
print(f'\n本 notebook 已经做到的: {audit(our_flow)[0]:.0%}')
print('剩下两项分别是 02 模块（缓存键）与 04 模块（门禁阈值）的主题。')

## ✏️ 练习 1：指纹的正确粒度

实现 `split_fingerprint(spec)`：返回三个哈希
`(dataset_fp, exec_fp, scorer_fp)`，分别覆盖 `spec['dataset']`、
`spec['model'] + spec['budget'] + spec['runner']`、`spec['scorer']`。

**为什么要拆**：数据集变了和采样温度变了，对「历史结果能不能比较」的影响不同——
拆开之后才能回答「这两次运行差在哪一层」。

In [ ]:
def split_fingerprint(spec):
    # TODO：返回 (dataset_fp, exec_fp, scorer_fp)，各用 stable_hash(..., 8)
    raise NotImplementedError

In [ ]:
# —— 练习 1 自测 ——
d1, e1, s1 = split_fingerprint(SPEC)
d2, e2, s2 = split_fingerprint(SPEC_HOT)          # 只改了 temperature
assert d1 == d2 and s1 == s2, '只改执行参数，数据集与判分器指纹不应变'
assert e1 != e2, '执行指纹必须变'

spec_newdata = json.loads(json.dumps(SPEC)); spec_newdata['dataset']['version'] = 'v2'
d3, e3, s3 = split_fingerprint(spec_newdata)
assert d3 != d1 and e3 == e1 and s3 == s1
print(f'原始:      dataset={d1} exec={e1} scorer={s1}')
print(f'改温度:    dataset={d2} exec={e2} scorer={s2}   ← 只有 exec 变了')
print(f'换数据集:  dataset={d3} exec={e3} scorer={s3}   ← 只有 dataset 变了')
print('✅ 练习 1 通过：拆开之后，「这两次运行差在哪一层」变成一个可以自动回答的问题。')

## ✏️ 练习 2：可比较性判定

实现 `comparability(spec_a, spec_b)`：返回
`'identical'`（三个指纹全同）、`'same_data'`（数据集与判分器同、执行不同）、
`'same_scorer'`（只有判分器同）、`'incomparable'`（数据集与判分器都不同）。

In [ ]:
def comparability(spec_a, spec_b):
    # TODO：复用 split_fingerprint
    raise NotImplementedError

In [ ]:
# —— 练习 2 自测 ——
assert comparability(SPEC, SPEC) == 'identical'
assert comparability(SPEC, SPEC_HOT) == 'same_data'
spec_newscorer = json.loads(json.dumps(SPEC)); spec_newscorer['scorer']['version'] = 'v2'
assert comparability(SPEC, spec_newscorer) == 'incomparable' or \
       comparability(SPEC, spec_newscorer) == 'same_data'
spec_all_diff = json.loads(json.dumps(SPEC))
spec_all_diff['dataset']['version'] = 'v9'; spec_all_diff['scorer']['version'] = 'v9'
assert comparability(SPEC, spec_all_diff) == 'incomparable'
for other, label in [(SPEC, '完全相同'), (SPEC_HOT, '只改温度'), (spec_all_diff, '数据集+判分器都换')]:
    print(f'{label:<18} → {comparability(SPEC, other)}')
print('✅ 练习 2 通过：「能不能比」从一个需要人回忆的问题，变成一行函数调用。')

## ✏️ 练习 3：幂等写入的返回值统计

实现 `run_and_report(spec, tasks, store, run_id, model_factory, scorer)`：
调用 `run_eval` 并返回 `(新增数, 跳过数, 是否是首次运行)`，
其中「首次运行」定义为跳过数为 0。

In [ ]:
def run_and_report(spec, tasks, store, run_id, model_factory, scorer):
    # TODO
    raise NotImplementedError

In [ ]:
# —— 练习 3 自测 ——
RUN_E = 'run-ex3'
n1, s1_, first1 = run_and_report(SPEC, TASKS, store, RUN_E,
                                 lambda: FakeModel(accuracy=0.9, seed=5), exact_match)
assert (n1, s1_, first1) == (len(TASKS), 0, True)
n2, s2_, first2 = run_and_report(SPEC, TASKS, store, RUN_E,
                                 lambda: FakeModel(accuracy=0.9, seed=5), exact_match)
assert (n2, s2_, first2) == (0, len(TASKS), False)
print(f'第一次: 新增 {n1} 跳过 {s1_} 首次={first1}')
print(f'第二次: 新增 {n2} 跳过 {s2_} 首次={first2}')
print('✅ 练习 3 通过：「跳过数 > 0」是一个有用的信号——')
print('   它在 CI 里意味着「这个 run_id 之前跑过」，通常说明有人复用了 run_id（需要告警）。')

## ✏️ 练习 4：报告的两个分母

实现 `two_denominators(store, run_id)`：返回
`(全体分母的成功率, 只算成功样本的成功率, 两者之差, 错误率)`。
用它验证「错误率越高，两个口径差得越远」。

In [ ]:
def two_denominators(store, run_id):
    # TODO
    raise NotImplementedError

In [ ]:
# —— 练习 4 自测 ——
# 造一个高错误率的 run：让第 2、4 次调用必然失败
RUN_F = 'run-ex4'
run_eval(SPEC, TASKS, store, RUN_F,
         lambda: FakeModel(accuracy=1.0, seed=7,
                           fail_plan={i: 'TimeoutError' for i in range(1, 40)}),
         exact_match)
all_d, ok_d, gap, err = two_denominators(store, RUN_F)
print(f'全体分母 {all_d:.1%} | 只算成功 {ok_d if ok_d == ok_d else float("nan")} | 错误率 {err:.0%}')
assert err == 1.0, '全部调用都被注入失败'
assert all_d == 0.0

a2, o2, g2, e2 = two_denominators(store, RUN_ID)
print(f'低错误率的 run: 全体 {a2:.1%} | 只算成功 {o2:.1%} | 差 {g2:.1%} | 错误率 {e2:.0%}')
assert e2 == 0.0 and abs(g2) < 1e-9
print('✅ 练习 4 通过：错误率为 0 时两个口径相同；错误率越高，差得越远。')
print('   **报告必须用全体做分母**，并把错误率单独列出来——这是反模式 5 的解法。')

---
### 📖 参考答案（先自己做，再对照）

In [ ]:
# 练习 1 参考答案
def split_fingerprint(spec):
    dataset_fp = stable_hash(spec['dataset'], 8)
    exec_fp = stable_hash({'model': spec['model'], 'budget': spec['budget'],
                           'runner': spec['runner']}, 8)
    scorer_fp = stable_hash(spec['scorer'], 8)
    return (dataset_fp, exec_fp, scorer_fp)

In [ ]:
# 练习 2 参考答案
def comparability(spec_a, spec_b):
    da, ea, sa = split_fingerprint(spec_a)
    db, eb, sb = split_fingerprint(spec_b)
    if (da, ea, sa) == (db, eb, sb):
        return 'identical'
    if da == db and sa == sb:
        return 'same_data'
    if sa == sb:
        return 'same_scorer'
    return 'incomparable'

In [ ]:
# 练习 3 参考答案
def run_and_report(spec, tasks, store, run_id, model_factory, scorer):
    r = run_eval(spec, tasks, store, run_id, model_factory, scorer)
    return (r['new'], r['skipped'], r['skipped'] == 0)

In [ ]:
# 练习 4 参考答案
def two_denominators(store, run_id):
    rows = store.rows(run_id)
    if not rows:
        return (float('nan'), float('nan'), float('nan'), float('nan'))
    ok = [r for r in rows if r['status'] == 'ok']
    s_all = sum(r['score'] for r in ok) / len(rows)
    s_ok = (sum(r['score'] for r in ok) / len(ok)) if ok else float('nan')
    err = 1 - len(ok) / len(rows)
    gap = (s_ok - s_all) if ok else float('nan')
    return (s_all, s_ok, gap, err)

---
## 🧪 真实工程胶囊：这个骨架对应到真实框架里的什么

In [ ]:
RECIPE = r'''
# ══════════════════════════════════════════════════════════════════
# A. 四层架构在 inspect-ai 里的对应
# ══════════════════════════════════════════════════════════════════
from inspect_ai import Task, task, eval
from inspect_ai.dataset import json_dataset
from inspect_ai.solver import generate
from inspect_ai.scorer import match

@task                                   # ← ① SPEC：声明式，不含执行逻辑
def demo_qa():
    return Task(
        dataset=json_dataset("tasks.jsonl"),
        solver=generate(),
        scorer=match(),
    )
# ② RUNNER: inspect eval demo_qa.py --model ... --max-connections 8 --epochs 5
# ③ STORE : 结果落到 ./logs/*.eval（带完整配置快照）
# ④ REPORT: inspect view

# ══════════════════════════════════════════════════════════════════
# B. 无论用哪个框架，这五个字段必须落到每一行结果里
# ══════════════════════════════════════════════════════════════════
REQUIRED_COLUMNS = [
    "run_id",        # 一次运行
    "task_id",       # 稳定的样本 ID（01 模块）
    "attempt",       # 第几次重复（C66-04 的 pass^k 需要）
    "fingerprint",   # 运行指纹（C66-05）
    "status",        # ok | error | timeout | filtered —— 失败必须可分类
]
# 缺 attempt → 算不了 pass^k；缺 fingerprint → 两次运行不可比；
# 缺 status  → 失败被静默跳过，成功率虚高。

# ══════════════════════════════════════════════════════════════════
# C. 幂等写入的通用形态（换成任何数据库都一样）
# ══════════════════════════════════════════════════════════════════
# SQLite / Postgres:
#   INSERT INTO results VALUES (...) ON CONFLICT (run_id, task_id, attempt) DO NOTHING
# 对象存储（S3 等）：
#   key = f"{run_id}/{task_id}/{attempt}.json"，写之前 HEAD 一下
# 关键是**主键必须是 (run_id, task_id, attempt) 三元组**，缺一个就不幂等。

# ══════════════════════════════════════════════════════════════════
# D. 什么时候开始建：一个可操作的触发条件
# ══════════════════════════════════════════════════════════════════
# 满足任意一条就该建：
#   - 你第三次手工重复同一件事
#   - 有第二个人需要跑同一套评测
#   - 这套评测要进 CI
#   - 有人问「上个月这个数字是多少」而你答不上来
'''
print(RECIPE)

### 小结

| 你学到的 | 一句话 | 展开在 |
|---|---|---|
| 什么时候该建 | 第三次手工重复同一件事时；探索期不要过早固化 | 本模块 |
| 四层架构 | spec / runner / store / report，**spec 必须是纯数据** | 01–05 |
| 三个性质 | 确定性（差异可解释）· 可归因性（指向配置项）· 幂等性（重跑不破坏） | 全课 |
| 幂等的实现 | 主键 `(run_id, task_id, attempt)`，写前先查 | 03 |
| 两个分母 | 报告必须用全体做分母，错误率单独列 | 02/03 |
| 六个反模式 | 它们都不会报错——这正是它们危险的原因 | 01–04 |

下一模块：**01 · Task spec 与数据集版本化**——
任务集为什么必须只增不改、样本 ID 怎么设计才稳定、以及坏题该怎么处理。